# Z-Score Based Network Anomaly Detection

## Objective

To detect anomalous network traffic using statistical Z-Score analysis and compare the detected anomalies with the ground-truth labels present in the CICIDS2017 dataset.

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from sklearn.metrics import classification_report

df = pd.read_csv("../data/ddos_clean.csv")

print(df.shape)
df.head()

(223082, 43)


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Bwd Packet Length Max,Bwd Packet Length Min,Flow Bytes/s,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Idle Mean,Idle Std,Idle Min,Label
0,54865,3,2,12,6,6,6.0,0,0,4.000000e+06,...,-1,1,20,0.0,0.0,0,0.0,0.0,0,BENIGN
1,55054,109,1,6,6,6,6.0,6,6,1.100917e+05,...,256,0,20,0.0,0.0,0,0.0,0.0,0,BENIGN
2,55055,52,1,6,6,6,6.0,6,6,2.307692e+05,...,256,0,20,0.0,0.0,0,0.0,0.0,0,BENIGN
3,46236,34,1,6,6,6,6.0,6,6,3.529412e+05,...,329,0,20,0.0,0.0,0,0.0,0.0,0,BENIGN
4,54863,3,2,12,6,6,6.0,0,0,4.000000e+06,...,-1,1,20,0.0,0.0,0,0.0,0.0,0,BENIGN


In [8]:
df["Anomaly"] = (
    df[" Label"] != "BENIGN"
).astype(int)

print(df["Anomaly"].value_counts())

print("Anomaly" in df.columns)
print(" Label" in df.columns)
print("Label" in df.columns)

Anomaly
1    128014
0     95068
Name: count, dtype: int64
True
True
False


In [ ]:
X = df.drop(
    [" Label", "Anomaly"],
    axis=1
)

y_true = df["Anomaly"]

#Scaling
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

z_scores = np.abs(X_scaled)

In [ ]:
threshold = 3  # If |z|>3 then anomaly

#Prediction
y_pred = (
    z_scores > threshold
).any(axis=1).astype(int)

print(pd.Series(y_pred).value_counts())

0    148809
1     74273
Name: count, dtype: int64


## Observation:
 Using a Z-score threshold of 3, the detector identified 74,273 flows as anomalous. This is substantially lower than the 128,014 attack flows present in the dataset, suggesting that a threshold of 3 may be too strict for this dataset and could result in missed detections.

In [ ]:
#Confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print(cm)

print(classification_report(y_true, y_pred))

[[ 45864  49204]
 [102945  25069]]
              precision    recall  f1-score   support

           0       0.31      0.48      0.38     95068
           1       0.34      0.20      0.25    128014

    accuracy                           0.32    223082
   macro avg       0.32      0.34      0.31    223082
weighted avg       0.33      0.32      0.30    223082



## Observation:
 Using a Z-score threshold of 3 resulted in low anomaly detection performance, achieving a recall of only 0.20 for DDoS traffic. This indicates that many attack flows do not exhibit sufficiently extreme feature values to be identified by a strict statistical threshold. Consequently, lower thresholds and more advanced anomaly detection techniques were investigated.